# Domain-Adaptive Colorization — Pipeline Runner

Runs `main.py` end to end: Grounded-SAM detection/routing → per-domain DDColor experts → soft blending → RGB reconstruction.

**Kaggle settings:** GPU accelerator (T4 x2 or P100) and **Internet ON**.

In [ ]:
# Cell 1: clone the project (branch B)
from pathlib import Path

REPO_DIR = Path("/kaggle/working/domain-adaptive-colorization")
if not REPO_DIR.exists():
    !git clone -b B https://github.com/zaeem886/domain-adaptive-colorization.git {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Cell 2: install all dependencies upfront safely
!pip install -q --no-deps lpips
!pip install -q segment-anything transformers timm einops scikit-image scipy opencv-python-headless huggingface-hub pyyaml tqdm pytest matplotlib pandas

In [ ]:
# Cell 3: run the test suite (no GPU, torch, cv2, or model weights required)
!PYTHONPATH=. python -m pytest -q tests

In [ ]:
# Cell 4: clone DDColor and download the pretrained DDColor-T checkpoint
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

DDCOLOR_DIR = Path("/kaggle/working/DDColor")
if not DDCOLOR_DIR.exists():
    !git clone --depth 1 https://github.com/piddnad/DDColor.git {DDCOLOR_DIR}

CHECKPOINT = hf_hub_download("piddnad/ddcolor_paper_tiny", "pytorch_model.bin")
print("DDColor:", DDCOLOR_DIR)
print("Checkpoint:", CHECKPOINT)

In [ ]:
# Cell 5: run the pipeline
IMAGE_PATH = "pics/dog.jpeg"          # change to your own image
OUTPUT_PATH = "outputs/kaggle_colorized.png"

!python main.py \
    --image {IMAGE_PATH} \
    --output {OUTPUT_PATH} \
    --single-checkpoint {CHECKPOINT} \
    --ddcolor-path {DDCOLOR_DIR} \
    --sam-checkpoint sam_vit_b.pth \
    --device cuda \
    --save-domain-map \
    --visualize \
    --stages-dir outputs/stages

In [ ]:
# Cell 6: display the intermediate pipeline stages
import os
from IPython.display import Image as IPImage, display

STAGES_DIR = "outputs/stages"
stages = [
    ("1. Decomposition", "01_decomposition.png"),
    ("2. SAM masks (raw + resolved)", "02_sam_masks.png"),
    ("3. Segments (crop + mask)", "03_segments.png"),
    ("4. Colorized segments", "04_colorized_segments.png"),
    ("5. Blended result", "05_blended.png"),
]
for title, filename in stages:
    path = os.path.join(STAGES_DIR, filename)
    print(title)
    if os.path.exists(path):
        display(IPImage(filename=path))
    else:
        print("  missing:", path)

In [ ]:
# Cell 7: display the input, result, and routed domain map
from IPython.display import Image as IPImage, display

print("Input")
display(IPImage(filename=IMAGE_PATH))
print("Colorized")
display(IPImage(filename=OUTPUT_PATH))
print("Domain map")
display(IPImage(filename=OUTPUT_PATH.replace(".png", "_domains.png")))

### Notes
- `--visualize` saves the intermediate stages to `outputs/stages/`: `01_decomposition.png`, `02_sam_masks.png`, `03_segments.png`, `04_colorized_segments.png`, `05_blended.png`, plus per-region files in `outputs/stages/regions/`.
- To skip detection and colorize the whole image with the general expert, add `--skip-detection` to the command above.
- For real per-domain experts, replace `--single-checkpoint` with `--checkpoints-dir checkpoints/`; the directory should contain `people.pth`, `nature.pth`, `urban.pth`, `animals.pth`, and `fallback.pth`.
- SAM ViT-B (~375 MB) is downloaded automatically on the first run if `sam_vit_b.pth` is missing.

In [ ]:
# Cell 9: Run Person C tests (metrics, plotting, test sets, E3 baseline)
!PYTHONPATH=. python -m pytest -q tests/test_metrics.py tests/test_plotting.py tests/test_test_sets.py tests/test_e3_baseline.py

In [ ]:
# Cell 9b: Validate the fixed COCO IDs and save an audit manifest
from pathlib import Path
from colorize.test_sets import validate_coco_ids, write_coco_metadata_manifest

annotation_candidates = [
    Path('/kaggle/input/coco-2017-dataset/coco2017/annotations/instances_val2017.json'),
    Path('/kaggle/input/coco-2017-dataset/annotations_trainval2017/annotations/instances_val2017.json'),
]
COCO_ANNOTATIONS = next((path for path in annotation_candidates if path.exists()), None)
if COCO_ANNOTATIONS is None:
    print('COCO annotations not mounted; ID validation was not run.')
else:
    validation = write_coco_metadata_manifest(
        COCO_ANNOTATIONS, 'outputs/coco_id_manifest.json'
    )
    print('COCO ID validation:', 'PASS' if validation['valid'] else 'FAIL')
    for domain, result in validation['domains'].items():
        print(domain, 'PASS' if result['valid'] else 'FAIL', result['mismatches'])
    if not validation['valid']:
        raise RuntimeError('Fixed test IDs failed COCO validation; inspect outputs/coco_id_manifest.json')

In [ ]:
# Cell 10: E3 - DDColor-T whole-image baseline on the domain set
import sys
sys.path.insert(0, str(DDCOLOR_DIR))

from colorize.test_sets import DOMAIN_SET_IDS, get_coco_image_path
from colorize.e3_baseline import run_e3_baseline
from pathlib import Path

# Point to your COCO val2017 images if available, else use the demo image
COCO_ROOT = Path("/kaggle/input/coco-2017-dataset/coco2017") if Path("/kaggle/input/coco-2017-dataset/coco2017").exists() else None

if COCO_ROOT:
    image_list = []
    for domain, ids in DOMAIN_SET_IDS.items():
        for img_id in ids:
            path = get_coco_image_path(img_id, COCO_ROOT)
            if Path(path).exists():
                image_list.append((img_id, path))
else:
    # Fallback: use the demo image
    image_list = [("demo", IMAGE_PATH)]

records = run_e3_baseline(
    image_paths=image_list,
    checkpoint=CHECKPOINT,
    ddcolor_path=str(DDCOLOR_DIR),
    output_csv="outputs/e3_results.csv",
    condition="ddcolor_t_whole",
    device="cuda",
    input_size=256,
    experiment="E3",
)
print(f"E3 baseline complete: {len(records)} records written to outputs/e3_results.csv")

In [ ]:
# Cell 11: Plot E3 results
from colorize.results import load_results
from colorize.plotting import plot_metric_bar, plot_metric_comparison
from pathlib import Path

records = load_results("outputs/e3_results.csv")
print(f"Loaded {len(records)} result records")

# Bar chart for colorfulness
fig = plot_metric_bar(records, metric="colorfulness", title="E3: Colorfulness (DDColor-T whole-image)", output_path="outputs/e3_colorfulness.png")
if fig:
    from IPython.display import Image as IPImage, display
    display(IPImage(filename="outputs/e3_colorfulness.png"))

# If PSNR is available, also plot it
psnr_records = [r for r in records if r.metric == "psnr"]
if psnr_records:
    fig2 = plot_metric_bar(records, metric="psnr", title="E3: PSNR (DDColor-T whole-image)", output_path="outputs/e3_psnr.png")
    if fig2:
        display(IPImage(filename="outputs/e3_psnr.png"))

### E3 Experiment Notes
- `condition="ddcolor_t_whole"` uses DDColor-T (256×256 input, ConvNeXt-T backbone) on the whole image.
- For DDColor-L, re-run Cell 11 with `input_size=512` and `condition="ddcolor_l_whole"`.
- If COCO val2017 is available as a Kaggle dataset, set `COCO_ROOT` appropriately.
- Results are appended to `outputs/e3_results.csv` in the format: experiment, condition, image_id, metric, value.
- Use `load_results("outputs/e3_results.csv")` and `plot_metric_comparison(...)` to compare both conditions.